# Interpolate QTRACK
The following code quickly interpolates 6hrly qtrack files to 3hrly to match the wrf files.

In [2]:
from datetime import datetime, timedelta

import numpy as np
import xarray as xr
import pandas as pd
import os
import glob
from netCDF4 import Dataset, num2date, date2num

from AEW_module import season, AEW, AEW_CCKW

from metpy.units import units
import geocat.viz as gv


import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import matplotlib.ticker as mticker 

import matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter
import wrf
from wrf import (to_np, interplevel, geo_bounds, getvar, smooth2d, get_cartopy, cartopy_xlim,
                 cartopy_ylim, latlon_coords, destagger)

import seaborn as sns
import metpy.calc as mpcalc

In [10]:
## The following are lists to select each ensemble set and respective initialization time
variation = ['fluxon', 'rst_on24', 'rst_on36', 'rst_on48', '', 'fluxoff', '', '', '']
init_times = ['0300','0303','0306',
             '0309','0312','0315',
             '0318','0321','0400']

In [16]:
### Pick ensemble set 
set_name = variation[0]     # try: variation[2]
ens = init_times[0]         # try: init_times[0]
wave = 'second'             # 'second' == Paulette or 'main' == Rene
title = 'Control'  # What will be displayed on main plot

save_name = set_name +"_"+ ens
print(save_name)

fluxon_0300


In [17]:
# From Anantha...
# Simple linear interpolation between each latitude point
def get_lats():
    new_lats = []
    wave_track_lats = df.lat[-len(times_list):]
    for i in range(0,len(wave_track_lats)):
        if wave_track_lats[i] == wave_track_lats[-1]:
            new_lats.append(wave_track_lats[i].values)
        else:
            next_value = 0.5*(wave_track_lats[i] + wave_track_lats[i+1])
            new_lats.append(wave_track_lats[i].values)
            new_lats.append(next_value.values)
    return new_lats

In [18]:
# From Anantha...
# Simple linear interpolation between each longitude point
def get_lons():
    new_lons = []
    wave_track_lons = df.lon[-len(times_list):] 
    for i in range(0,len(wave_track_lons)):
        if wave_track_lons[i] == wave_track_lons[-1]:
            new_lons.append(wave_track_lons[i].values)
        else:
            next_value = 0.5*(wave_track_lons[i] + wave_track_lons[i+1])
            new_lons.append(wave_track_lons[i].values)
            new_lons.append(next_value.values)
    return new_lons

In [19]:
# Get track data
start_time ='2020-09-'+ens[:2]+' '+ens[2:]+':00'

# QTRACK data ends on the 9th for odd ens members
if int(ens) % 2 == 0:
    end_time = '2020-09-09 12:00'
    print('even ens member. ending at 12z')
else:
    end_time = '2020-09-09 09:00'
    print('odd ens member. ending at 09z')
    
# This is for the QTRACK data
times_list = pd.date_range(start=start_time, end=end_time, freq='6h') # Our QTRACK data is 6hrly
df = xr.open_dataset('/glade/u/home/athornton/qtrack/'+wave+'_wave/'+wave+'_wave_track_'+save_name+'.nc')
# Interpolate lats and lons to 3 hours
wave_track_lats = get_lats()
wave_track_lons = get_lons()

even ens member. ending at 12z


In [21]:
len(wave_track_lats)

53

In [22]:
len(wave_track_lons)

53